# Build Silver & Gold datasets

Ce notebook montre comment reconstruire le dataset match-level à partir des CSV bronze (games/boxscores), puis enchaîner sur les builds silver/gold via `build_silver_and_gold`. Ajuste simplement les drapeaux ci-dessous pour utiliser un fichier existant ou recalculer depuis les bronzes.

In [1]:
from pathlib import Path

import pandas as pd

from src.datasets import build_match_dataset_from_bronze, build_silver_and_gold

# Config
BUILD_FROM_BRONZE = True  # Passe à False si tu veux charger un fichier existant
BRONZE_MATCHES_PATH = Path("data/01_bronze/matches/bronze_matches_20251109T143558Z.parquet")
TARGET = "POINT_TOTAL"  # IS_WIN, POINT_DIFF, POINT_TOTAL, ...
#TARGET = "IS_WIN"  # IS_WIN, POINT_DIFF, POINT_TOTAL, ...

if BUILD_FROM_BRONZE:
    bronze_result = build_match_dataset_from_bronze()
    bronze_matches_df = bronze_result.dataset
    print("Bronze matches metadata:", bronze_result.metadata)
    print("Bronze matches artifact:", bronze_result.path)
else:
    if BRONZE_MATCHES_PATH.suffix.lower() in {".parquet", ".pq"}:
        bronze_matches_df = pd.read_parquet(BRONZE_MATCHES_PATH)
    else:
        bronze_matches_df = pd.read_csv(BRONZE_MATCHES_PATH)

silver_result, gold_result = build_silver_and_gold(bronze_matches_df, target=TARGET)

print("Silver metadata:", silver_result.metadata)
print("Silver artifact:", silver_result.path)
print("Gold metadata:", gold_result.metadata)
print("Gold artifact:", gold_result.path)


/home/ju/Documents/Dev/NBA_Predictor/src/feature_aggregation.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tops = roll.groupby(['GAME_ID', 'TEAM_ID']).apply(pick_top).reset_index()


------------------ Nombre de lignes sans cotes (home/away): 64564 ------------------
[reshaping] dropping 12 duplicate away rows based on match keys ['GAME_ID', 'GAME_DATE', 'SEASON'].


/home/ju/Documents/Dev/NBA_Predictor/src/features/reshaping.py:149: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[diff_col] = df[home_col] - df[away_col]
/home/ju/Documents/Dev/NBA_Predictor/src/features/reshaping.py:149: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[diff_col] = df[home_col] - df[away_col]
/home/ju/Documents/Dev/NBA_Predictor/src/features/reshaping.py:149: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

Bronze matches metadata: {'rows': 32276, 'columns': 341, 'games_source': 'games_merged_all_seasons_20251111T203614Z.csv', 'boxscores_source': 'all_seasons_boxscores_merged_20251111T203614Z.csv'}
Bronze matches artifact: data/01_bronze/matches/bronze_matches_20251111T233719Z.parquet


/home/ju/Documents/Dev/NBA_Predictor/src/features/reshaping.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base[f"OPP_{col}"] = match_df[opp_src]
/home/ju/Documents/Dev/NBA_Predictor/src/features/reshaping.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base[col] = match_df[src]
/home/ju/Documents/Dev/NBA_Predictor/src/features/reshaping.py:90: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns 

Silver metadata: {'rows': 32276, 'columns': 4033, 'feature_plan': {'feature_steps': [{'name': 'team_history', 'stage': 'team_features', 'description': 'Convert match rows to team view, compute rolling stats (rest, win, H2H, Elo).', 'targets': ['IS_WIN', 'POINT_DIFF', 'POINT_TOTAL']}, {'name': 'availability_rollups', 'stage': 'availability_features', 'description': 'Leak-safe rolling absence/injury rates for key players.', 'targets': ['IS_WIN', 'POINT_DIFF', 'POINT_TOTAL']}, {'name': 'matchup_scoring', 'stage': 'matchup_features', 'description': 'Combine home/away rolling stats into matchup-level pace, total, and gap metrics.', 'targets': ['IS_WIN', 'POINT_DIFF', 'POINT_TOTAL']}, {'name': 'drop_raw_team_columns', 'stage': 'cleanup', 'description': 'Remove raw per-game stat columns once rollups exist.', 'targets': ['IS_WIN', 'POINT_DIFF', 'POINT_TOTAL']}, {'name': 'drop_helper_columns', 'stage': 'cleanup', 'description': 'Remove helper columns used only during feature construction.', 'ta

## Notes

- Utilise le CLI `python -m src.datasets.build --build-from-bronze --stage all --target IS_WIN` pour lancer le même flux depuis le terminal (ou `--matches chemin.parquet` si tu as déjà un dataset match-level).
- Les artefacts sont écrits par défaut en Parquet dans `data/01_bronze/matches`, `data/02_silver` et `data/03_gold`. Passe `--no-save` coté CLI (ou `persist_artifact=False` dans les configs) si tu veux seulement un build en mémoire.